In [1]:
import requests
import re
import time
import json
import argparse
import pandas as pd

In [2]:
def query_vllm_api(server_url: str, prompt: str, temperature: float, max_tokens: int):
    """ Sends a request to a vLLM server and returns the response and elapsed time. """
    payload = {
        "model": "Qwen/Qwen2.5-1.5B-Instruct",
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": temperature
    }
    headers = {"Content-Type": "application/json"}

    try:
        start_time = time.time()
        response = requests.post(f"{server_url}/v1/completions", headers=headers, data=json.dumps(payload))
        end_time = time.time()
        response.raise_for_status()
        return response.json()["choices"][0]["text"], end_time - start_time
    except requests.exceptions.RequestException as e:
        print(f"Error querying vLLM API at {server_url}: {e}")
        return None, None

In [3]:
def parse_prometheus_metrics(prometheus_text: str):
    """
    Parses Prometheus-style text metrics into a dictionary.
    """
    metrics = {}

    for line in prometheus_text.splitlines():
        if line.startswith("#"):
            continue  # Skip comments and HELP/TYPE metadata

        match = re.match(r'([\w:]+)\{([^}]*)\}\s+([\d.e+-]+)', line)
        if match:
            metric_name, labels, value = match.groups()
            metrics[metric_name] = float(value)
        else:
            # Handle simple metric lines without labels
            parts = line.split()
            if len(parts) == 2:
                metric_name, value = parts
                metrics[metric_name] = float(value)

    return metrics

In [4]:
def get_vllm_metrics(server_url: str):
    """
    Fetches and parses vLLM metrics from the /metrics endpoint.
    """
    try:
        response = requests.get(f"{server_url}/metrics")
        response.raise_for_status()

        if not response.text.strip():
            print(f"Error: Empty response from {server_url}/metrics.")
            return None

        metrics = parse_prometheus_metrics(response.text)

        # Get token counts
        prompt_tokens = metrics.get("vllm:prompt_tokens_total", 0.0)
        generation_tokens = metrics.get("vllm:generation_tokens_total", 0.0)

        # Get total time for processing requests
        time_to_first_token = metrics.get("vllm:time_to_first_token_seconds_sum", 0.0)
        total_generation_time = metrics.get("vllm:time_per_output_token_seconds_sum", 0.0)

        # Compute throughput
        avg_prompt_throughput = prompt_tokens / time_to_first_token if time_to_first_token > 0 else 0.0
        avg_generation_throughput = generation_tokens / total_generation_time if total_generation_time > 0 else 0.0

        return {
            "num_requests_running": metrics.get("vllm:num_requests_running", 0.0),
            "num_requests_swapped": metrics.get("vllm:num_requests_swapped", 0.0),
            "num_requests_waiting": metrics.get("vllm:num_requests_waiting", 0.0),
            "gpu_cache_usage": metrics.get("vllm:gpu_cache_usage_perc", 0.0),
            "cpu_cache_usage": metrics.get("vllm:cpu_cache_usage_perc", 0.0),
            "prefix_cache_hit_rate_gpu": metrics.get("vllm:gpu_prefix_cache_hit_rate", 0.0),
            "prefix_cache_hit_rate_cpu": metrics.get("vllm:cpu_prefix_cache_hit_rate", 0.0),
            "prompt_tokens_total": prompt_tokens,
            "generation_tokens_total": generation_tokens,
            "avg_prompt_throughput": avg_prompt_throughput,
            "avg_generation_throughput": avg_generation_throughput,
            "time_to_first_token": time_to_first_token,
            "time_per_output_token": total_generation_time,
            "e2e_request_latency": metrics.get("vllm:e2e_request_latency_seconds_sum", 0.0),
        }

    except requests.exceptions.RequestException as e:
        print(f"Error fetching vLLM metrics from {server_url}: {e}")
        return None

In [5]:
def benchmark_vllm(server_url: str, prompt, temperature, max_tokens, repeat_count):
    """ Runs a benchmark by querying the vLLM API multiple times and measuring latency. """
    print(f"Benchmarking vLLM server at {server_url}...\n")

    total_time = 0
    metrics_list = []  # Store all iteration metrics

    for i in range(repeat_count):
        response, elapsed_time = query_vllm_api(server_url, prompt, temperature, max_tokens)

        if response:
            total_time += elapsed_time

        # Fetch performance metrics
        metrics = get_vllm_metrics(server_url)
        if metrics:
            metrics_list.append({
                "Iteration": i + 1,
                "Prompt Thruput (tok/s)": round(metrics['avg_prompt_throughput'], 2),
                "Gen Thruput (tok/s)": round(metrics['avg_generation_throughput'], 2),
                "GPU Cache (%)": round(metrics['gpu_cache_usage'] * 100, 2),
                "CPU Cache (%)": round(metrics['cpu_cache_usage'] * 100, 2),
                "Prefix Hit Rate (GPU) (%)": round(metrics['prefix_cache_hit_rate_gpu'] * 100, 2),
                "Prefix Hit Rate (CPU) (%)": round(metrics['prefix_cache_hit_rate_cpu'] * 100, 2),
            })

    # Convert list to a pandas DataFrame
    df = pd.DataFrame(metrics_list)

    # Print the DataFrame in a readable way
    print("\n===== Benchmark Results =====")
    print(df.to_string(index=False))  # Prevents line breaks if terminal width is limited
    
    # Print final average request time
    avg_time = total_time / repeat_count
    print(f"\nAverage time per request: {avg_time:.3f} seconds\n")

In [6]:
def main(server_url="http://localhost:8000", prompt="Hello world", repeat_count=10, temperature=0, max_tokens=2):
    benchmark_vllm(server_url, prompt, temperature, max_tokens, repeat_count)

In [7]:
LONG_PROMPT = "You are a helpful assistant in recognizes the content of tables in markdown format. Here is a table as follows.\n# Table\n" + """
| ID  | Name          | Age | Occupation    | Country       | Email                  | Phone Number   | Address                       |
|-----|---------------|-----|---------------|---------------|------------------------|----------------|------------------------------|
| 1   | John Doe      | 29  | Engineer      | USA           | john.doe@example.com   | 555-1234       | 123 Elm St, Springfield, IL  |
| 2   | Jane Smith    | 34  | Doctor        | Canada        | jane.smith@example.com | 555-5678       | 456 Oak St, Toronto, ON      |
| 3   | Alice Johnson | 27  | Teacher       | UK            | alice.j@example.com    | 555-8765       | 789 Pine St, London, UK      |
| 4   | Bob Brown     | 45  | Artist        | Australia     | bob.b@example.com      | 555-4321       | 321 Maple St, Sydney, NSW    |
| 5   | Carol White   | 31  | Scientist     | New Zealand   | carol.w@example.com    | 555-6789       | 654 Birch St, Wellington, NZ |
| 6   | Dave Green    | 28  | Lawyer        | Ireland       | dave.g@example.com     | 555-3456       | 987 Cedar St, Dublin, IE     |
| 7   | Emma Black    | 40  | Musician      | USA           | emma.b@example.com     | 555-1111       | 246 Ash St, New York, NY     |
| 8   | Frank Blue    | 37  | Chef          | Canada        | frank.b@example.com    | 555-2222       | 135 Spruce St, Vancouver, BC |
| 9   | Grace Yellow  | 50  | Engineer      | UK            | grace.y@example.com    | 555-3333       | 864 Fir St, Manchester, UK   |
| 10  | Henry Violet  | 32  | Artist        | Australia     | henry.v@example.com    | 555-4444       | 753 Willow St, Melbourne, VIC|
| 11  | Irene Orange  | 26  | Scientist     | New Zealand   | irene.o@example.com    | 555-5555       | 912 Poplar St, Auckland, NZ  |
| 12  | Jack Indigo   | 38  | Teacher       | Ireland       | jack.i@example.com     | 555-6666       | 159 Elm St, Cork, IE         |
| 13  | Karen Red     | 41  | Lawyer        | USA           | karen.r@example.com    | 555-7777       | 357 Cedar St, Boston, MA     |
| 14  | Leo Brown     | 30  | Chef          | Canada        | leo.b@example.com      | 555-8888       | 246 Oak St, Calgary, AB      |
| 15  | Mia Green     | 33  | Musician      | UK            | mia.g@example.com      | 555-9999       | 975 Pine St, Edinburgh, UK   |
| 16  | Noah Yellow   | 29  | Doctor        | Australia     | noah.y@example.com     | 555-0000       | 864 Birch St, Brisbane, QLD  |
| 17  | Olivia Blue   | 35  | Engineer      | New Zealand   | olivia.b@example.com   | 555-1212       | 753 Maple St, Hamilton, NZ   |
| 18  | Peter Black   | 42  | Artist        | Ireland       | peter.b@example.com    | 555-3434       | 912 Fir St, Limerick, IE     |
| 19  | Quinn White   | 28  | Scientist     | USA           | quinn.w@example.com    | 555-5656       | 159 Willow St, Seattle, WA   |
| 20  | Rachel Red    | 31  | Teacher       | Canada        | rachel.r@example.com   | 555-7878       | 357 Poplar St, Ottawa, ON    |
| 21  | Steve Green   | 44  | Lawyer        | UK            | steve.g@example.com    | 555-9090       | 753 Elm St, Birmingham, UK   |
| 22  | Tina Blue     | 36  | Musician      | Australia     | tina.b@example.com     | 555-1213       | 864 Cedar St, Perth, WA      |
| 23  | Umar Black    | 39  | Chef          | New Zealand   | umar.b@example.com     | 555-3435       | 975 Spruce St, Christchurch, NZ|
| 24  | Victor Yellow | 43  | Engineer      | Ireland       | victor.y@example.com   | 555-5657       | 246 Willow St, Galway, IE    |
| 25  | Wendy Orange  | 27  | Artist        | USA           | wendy.o@example.com    | 555-7879       | 135 Elm St, Denver, CO       |
| 26  | Xavier Green  | 34  | Scientist     | Canada        | xavier.g@example.com   | 555-9091       | 357 Oak St, Montreal, QC     |
| 27  | Yara Red      | 41  | Teacher       | UK            | yara.r@example.com     | 555-1214       | 975 Pine St, Leeds, UK       |
| 28  | Zack Blue     | 30  | Lawyer        | Australia     | zack.b@example.com     | 555-3436       | 135 Birch St, Adelaide, SA   |
| 29  | Amy White     | 33  | Musician      | New Zealand   | amy.w@example.com      | 555-5658       | 159 Maple St, Wellington, NZ |
| 30  | Ben Black     | 38  | Chef          | Ireland       | ben.b@example.com      | 555-7870       | 246 Fir St, Waterford, IE    |
"""

main(server_url="http://localhost:8000")

In [9]:
main(server_url="http://localhost:8000", prompt=LONG_PROMPT)

Benchmarking vLLM server at http://localhost:8000...


===== Benchmark Results =====
 Iteration  Prompt Thruput (tok/s)  Gen Thruput (tok/s)  GPU Cache (%)  CPU Cache (%)  Prefix Hit Rate (GPU) (%)  Prefix Hit Rate (CPU) (%)
         1                   47.51                12.36            0.0            0.0                      87.62                        0.0
         2                   47.55                12.36            0.0            0.0                      87.62                        0.0
         3                   47.58                12.36            0.0            0.0                      87.63                        0.0
         4                   47.61                12.36            0.0            0.0                      87.64                        0.0
         5                   47.64                12.36            0.0            0.0                      87.65                        0.0
         6                   47.67                12.36            0.0     